# Projeto Integrador G4 — Regressão Linear
## Secretaria de Cultura de São Paulo

## Ideia do caso de uso

Neste notebook vamos trabalhar com o dataset de eventos e contratações da **Secretaria de Cultura de São Paulo**.

O objetivo é construir um pipeline completo de ciência de dados e compará-lo em **três variantes**
do dataset, separadas por **categoria de artista** (faixa de cachê `Valor_dia`):

| Variante | Categoria | Perfil do artista | Faixa de cachê (`Valor_dia`) | Arquivo |
|---|---|---|---|---|
| **G1** | `Categoria_Id == 1` | **Pequenos Artistas** | R$ 0,00 a R$ 9.999,99 | `Dataset_Secretaria_Cultura_SP_varG1.csv` |
| **G2** | `Categoria_Id == 2` | **Artistas Médios** | R$ 10.000,00 a R$ 49.999,99 | `Dataset_Secretaria_Cultura_SP_varG2.csv` |
| **G3** | `Categoria_Id == 3` | **Super Artistas** | acima de R$ 50.000,00 | `Dataset_Secretaria_Cultura_SP_varG3.csv` |

Para cada variante o pipeline executa as mesmas etapas:

1. **Explorar** os dados;
2. **Detectar anomalias**;
3. **Tratar** o dataset;
4. **Preparar** as variáveis para modelagem;
5. **Treinar** uma **Regressão Linear** para prever o `Valor_dia` dos eventos;
6. **Avaliar** a eficácia do treinamento comparando treino e teste.

Ao final, **comparamos as três variantes** (R², MAE e RMSE) para entender em qual categoria a
Regressão Linear funciona melhor.

## Observação importante

A **Regressão Linear** pressupõe relações aproximadamente lineares entre as variáveis explicativas e a variável-alvo.

Por isso, além do treinamento, também analisaremos:

- resíduos;
- comparação entre valores reais e previstos;
- diferença de desempenho entre treino e teste.

## Esquema de modelagem (igual nas 3 variantes)

Como cada arquivo já contém apenas uma `Categoria_Id`, essa coluna é constante e é **removida**
(assim como a coluna `Id`). O alvo é `Valor_dia` e as features são:

- *target encoding* na coluna de alta cardinalidade (`Fornecedor_Id`);
- *one-hot encoding* nas categóricas de baixa cardinalidade (`Regiao_Id`, `Zona_Id`);
- variáveis numéricas: `Ano` e `Mes`.

## Saídas geradas

Cada variante produz um PDF `G1_pipeline.pdf`, `G2_pipeline.pdf` e `G3_pipeline.pdf` em `Relatorios/`.
A comparação final gera `00_comparativo.pdf` e tudo é unido em `00_consolidado.pdf`.

## 1. Importação das bibliotecas

Vamos usar bibliotecas para:

- manipulação tabular com `pandas`;
- cálculos numéricos com `numpy`;
- visualização com `matplotlib` e `seaborn`;
- padronização com `StandardScaler`;
- detecção de anomalias com `KMeans`;
- modelagem supervisionada com `LinearRegression`;
- geração de relatórios em PDF com `PdfPages`.

In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from pypdf import PdfReader, PdfWriter
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)

# ---------------------------------------------------------------------------
# Caminhos do projeto
# Funciona tanto executando da pasta Codigo/ quanto da raiz do projeto
# ---------------------------------------------------------------------------
def encontrar_base_dir():
    candidatos = [Path.cwd(), Path.cwd().parent, Path('..').resolve(), Path('.').resolve()]
    vistos = set()
    for candidato in candidatos:
        candidato = candidato.resolve()
        if candidato in vistos:
            continue
        vistos.add(candidato)
        if (candidato / 'Datasets' / 'Dataset_Secretaria_Cultura_SP_varG1.csv').exists():
            return candidato
    raise FileNotFoundError(
        'Datasets não encontrados. Execute o notebook a partir da pasta Codigo/ ou da raiz do projeto.'
    )

BASE_DIR = encontrar_base_dir()
DATASETS_DIR = BASE_DIR / 'Datasets'
RELATORIOS_DIR = BASE_DIR / 'Relatorios'
RELATORIOS_DIR.mkdir(parents=True, exist_ok=True)

# Parâmetros globais do experimento
RANDOM_STATE = 42
TEST_SIZE = 0.15
PERCENTIL_ANOMALIA = 95
N_CLUSTERS_ANOMALIA = 4
TARGET_COL = 'Valor_dia'

# ---------------------------------------------------------------------------
# Variantes a comparar: G1, G2 e G3 (Categoria_Id 1, 2 e 3)
# ---------------------------------------------------------------------------
VARIANTES = {
    'G1': DATASETS_DIR / 'Dataset_Secretaria_Cultura_SP_varG1.csv',
    'G2': DATASETS_DIR / 'Dataset_Secretaria_Cultura_SP_varG2.csv',
    'G3': DATASETS_DIR / 'Dataset_Secretaria_Cultura_SP_varG3.csv',
}

# ---------------------------------------------------------------------------
# Legenda das categorias de artista (faixa de cachê Valor_dia)
# ---------------------------------------------------------------------------
PERFIS = {
    'G1': {'categoria': 1, 'nome': 'Pequenos Artistas', 'faixa': 'R$ 0,00 a R$ 9.999,99'},
    'G2': {'categoria': 2, 'nome': 'Artistas Médios', 'faixa': 'R$ 10.000,00 a R$ 49.999,99'},
    'G3': {'categoria': 3, 'nome': 'Super Artistas', 'faixa': 'acima de R$ 50.000,00'},
}

print(f'Projeto: {BASE_DIR.name}')
print('Variantes a executar:')
for nome, caminho in VARIANTES.items():
    perfil = PERFIS[nome]
    print(f'  {nome}: {perfil["nome"]} ({perfil["faixa"]}) -> {caminho.name} (existe: {caminho.exists()})')
print(f'Pasta de relatórios: {RELATORIOS_DIR}')

Projeto: Proj_Integrador_G4
Variantes a executar:
  G1: Pequenos Artistas (R$ 0,00 a R$ 9.999,99) -> Dataset_Secretaria_Cultura_SP_varG1.csv (existe: True)
  G2: Artistas Médios (R$ 10.000,00 a R$ 49.999,99) -> Dataset_Secretaria_Cultura_SP_varG2.csv (existe: True)
  G3: Super Artistas (acima de R$ 50.000,00) -> Dataset_Secretaria_Cultura_SP_varG3.csv (existe: True)
Pasta de relatórios: C:\Source\PUCSP\Proj_Integrador_G4\Relatorios


## 2. Funções auxiliares

Antes de iniciar a análise, vamos organizar funções reutilizáveis.

Isso facilita:

- manter o notebook didático;
- gerar PDFs por etapa;
- reaproveitar exatamente o mesmo pipeline nas três variantes (G1, G2 e G3).

In [2]:
def salvar_pagina_texto(pdf, titulo, linhas, subtitulo=None):
    """Salva uma página textual no PDF da etapa."""
    fig, ax = plt.subplots(figsize=(8.27, 11.69))
    ax.axis('off')

    y = 0.95
    ax.text(0.05, y, titulo, fontsize=16, fontweight='bold', va='top')
    y -= 0.05

    if subtitulo:
        ax.text(0.05, y, subtitulo, fontsize=11, va='top', color='dimgray')
        y -= 0.05

    for linha in linhas:
        ax.text(0.05, y, linha, fontsize=10, va='top', family='monospace')
        y -= 0.035
        if y < 0.05:
            break

    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)


def salvar_tabela_pdf(pdf, titulo, dataframe, max_linhas=20):
    """Salva uma tabela formatada como página do PDF."""
    df_mostra = dataframe.head(max_linhas).copy()

    fig, ax = plt.subplots(figsize=(11.69, 8.27))
    ax.axis('off')
    ax.set_title(titulo, fontsize=14, fontweight='bold', loc='left', pad=20)

    tabela = ax.table(
        cellText=df_mostra.values,
        colLabels=df_mostra.columns,
        loc='center',
        cellLoc='center',
    )
    tabela.auto_set_font_size(False)
    tabela.set_fontsize(8)
    tabela.scale(1, 1.4)

    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)


def consolidar_pdfs(caminhos_pdf, caminho_saida):
    """Junta vários PDFs em um único arquivo consolidado."""
    writer = PdfWriter()

    for caminho in caminhos_pdf:
        if not caminho.exists():
            print(f'Aviso: PDF não encontrado -> {caminho.name}')
            continue
        reader = PdfReader(str(caminho))
        for pagina in reader.pages:
            writer.add_page(pagina)

    with open(caminho_saida, 'wb') as arquivo:
        writer.write(arquivo)

    print(f'PDF consolidado salvo em: {caminho_saida.name}')


def detectar_anomalias_iqr(serie, fator=1.5):
    """
    Detecta outliers pelo método IQR.

    Regra didática:
    - Q1 = percentil 25%
    - Q3 = percentil 75%
    - IQR = Q3 - Q1
    - outlier se valor < Q1 - 1.5 * IQR ou > Q3 + 1.5 * IQR
    """
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - fator * iqr
    limite_superior = q3 + fator * iqr

    mascara = (serie < limite_inferior) | (serie > limite_superior)
    return mascara, limite_inferior, limite_superior, q1, q3, iqr


def detectar_anomalias_kmeans(df, colunas, n_clusters=4, percentil=95, random_state=42):
    """
    Usa distância ao centróide do K-Means como score de anomalia.

    Observação didática:
    K-Means não é nativo para detecção de anomalias, mas é útil para ensinar
    a lógica de distância a padrões aprendidos.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[colunas])

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    kmeans.fit(X_scaled)

    distancias = kmeans.transform(X_scaled).min(axis=1)
    limiar = np.percentile(distancias, percentil)

    return distancias, limiar, kmeans, scaler


def tratar_ausentes(df):
    """Preenche ausentes com mediana (numéricas) ou moda (categóricas)."""
    df_limpo = df.copy()
    relatorio = []

    for coluna in df_limpo.columns:
        nulos = df_limpo[coluna].isnull().sum()
        if nulos == 0:
            continue

        if pd.api.types.is_numeric_dtype(df_limpo[coluna]):
            valor = df_limpo[coluna].median()
            estrategia = 'mediana'
        else:
            valor = df_limpo[coluna].mode(dropna=True).iloc[0]
            estrategia = 'moda'

        df_limpo[coluna] = df_limpo[coluna].fillna(valor)
        relatorio.append({'coluna': coluna, 'nulos': nulos, 'estrategia': estrategia, 'valor': valor})

    return df_limpo, pd.DataFrame(relatorio)


def target_encode_coluna(serie_treino, y_treino, serie_teste, smoothing=10):
    """
    Target encoding com suavização.

    A média global entra no cálculo para reduzir overfitting em categorias raras.
    """
    media_global = float(y_treino.mean())
    cat_treino = serie_treino.astype(str)
    cat_teste = serie_teste.astype(str)

    stats = (
        pd.DataFrame({'cat': cat_treino, 'target': y_treino})
        .groupby('cat')['target']
        .agg(['mean', 'count'])
    )

    stats['encoded'] = (
        (stats['count'] * stats['mean'] + smoothing * media_global)
        / (stats['count'] + smoothing)
    )

    treino_encoded = cat_treino.map(stats['encoded']).astype(float).fillna(media_global)
    teste_encoded = cat_teste.map(stats['encoded']).astype(float).fillna(media_global)
    return treino_encoded, teste_encoded


def montar_dataset_modelagem(df, target_col='Valor_dia', test_size=0.15, random_state=42):
    """
    Prepara X e y para a Regressão Linear (esquema único das variantes G1, G2 e G3).

    - colunas removidas: Id e Categoria_Id (constante dentro de cada variante)
    - colunas numéricas: Ano e Mes
    - target encoding em Fornecedor_Id
    - one-hot em Regiao_Id e Zona_Id
    - alvo: Valor_dia
    """
    colunas_numericas = ['Ano', 'Mes']
    colunas_onehot = ['Regiao_Id', 'Zona_Id']
    colunas_target_encode = ['Fornecedor_Id']

    df_model = df.drop(columns=['Id', 'Categoria_Id'], errors='ignore').copy()
    y = df_model[target_col].astype(float)
    X_base = df_model.drop(columns=[target_col])

    X_treino_base, X_teste_base, y_treino, y_teste = train_test_split(
        X_base,
        y,
        test_size=test_size,
        random_state=random_state,
    )

    partes_treino = [X_treino_base[colunas_numericas].reset_index(drop=True)]
    partes_teste = [X_teste_base[colunas_numericas].reset_index(drop=True)]

    for coluna in colunas_target_encode:
        tr_enc, te_enc = target_encode_coluna(
            X_treino_base[coluna].reset_index(drop=True),
            y_treino.reset_index(drop=True),
            X_teste_base[coluna].reset_index(drop=True),
        )
        partes_treino.append(tr_enc.rename(f'{coluna}_te'))
        partes_teste.append(te_enc.rename(f'{coluna}_te'))

    X_treino_oh = pd.get_dummies(
        X_treino_base[colunas_onehot].astype(str),
        columns=colunas_onehot,
        drop_first=False,
    )
    X_teste_oh = pd.get_dummies(
        X_teste_base[colunas_onehot].astype(str),
        columns=colunas_onehot,
        drop_first=False,
    )
    X_teste_oh = X_teste_oh.reindex(columns=X_treino_oh.columns, fill_value=0)

    partes_treino.append(X_treino_oh.reset_index(drop=True))
    partes_teste.append(X_teste_oh.reset_index(drop=True))

    X_treino = pd.concat(partes_treino, axis=1)
    X_teste = pd.concat(partes_teste, axis=1)

    return X_treino, X_teste, y_treino, y_teste


def avaliar_regressao(modelo, X_treino, X_teste, y_treino, y_teste):
    """Calcula métricas de regressão no treino e no teste."""
    y_pred_treino = modelo.predict(X_treino)
    y_pred_teste = modelo.predict(X_teste)

    metricas = {
        'conjunto': ['Treino', 'Teste'],
        'R2': [
            r2_score(y_treino, y_pred_treino),
            r2_score(y_teste, y_pred_teste),
        ],
        'MAE': [
            mean_absolute_error(y_treino, y_pred_treino),
            mean_absolute_error(y_teste, y_pred_teste),
        ],
        'RMSE': [
            np.sqrt(mean_squared_error(y_treino, y_pred_treino)),
            np.sqrt(mean_squared_error(y_teste, y_pred_teste)),
        ],
    }

    df_metricas = pd.DataFrame(metricas)
    return df_metricas, y_pred_treino, y_pred_teste


print('Funções auxiliares carregadas com sucesso.')

Funções auxiliares carregadas com sucesso.


## 3. Pipeline completo por variante

A função `executar_pipeline_completo` roda, para **uma** variante, todas as etapas em sequência:

1. **Exploração** — estatísticas e distribuição de `Valor_dia`;
2. **Anomalias** — IQR + distância ao centróide do K-Means (combinadas por OU);
3. **Tratamento** — remoção de anomalias, tratamento de ausentes e cópia limpa do dataset;
4. **Preparação** — `train_test_split`, target/one-hot encoding e `StandardScaler` (fit só no treino);
5. **Treino** — `LinearRegression` e análise dos coeficientes;
6. **Avaliação** — R², MAE, RMSE no treino e no teste, resíduos e real vs. previsto.

Cada chamada gera um PDF próprio (`G1_pipeline.pdf`, etc.) e retorna um dicionário com as
métricas usadas na comparação final.

In [3]:
def executar_pipeline_completo(nome, dataset_path):
    """Roda o pipeline completo (etapas 1 a 6) para uma variante e gera o PDF da variante."""
    print(f'\n========================= Variante {nome} =========================')

    # -----------------------------------------------------------------------
    # Etapa 1 — Carga e exploração
    # ATENÇÃO: o arquivo usa separador ';' e decimal ',' (padrão brasileiro)
    # -----------------------------------------------------------------------
    df = pd.read_csv(dataset_path, sep=';', decimal=',')
    categoria = int(df['Categoria_Id'].iloc[0]) if 'Categoria_Id' in df.columns else None
    perfil = PERFIS.get(nome, {'nome': f'Variante {nome}', 'faixa': ''})
    rotulo = f'{nome} — {perfil["nome"]}'
    print(f'[{nome}] {perfil["nome"]} ({perfil["faixa"]}) | Registros: {df.shape[0]:,} | Categoria_Id: {categoria}')

    pdf_path = RELATORIOS_DIR / f'{nome}_pipeline.pdf'
    with PdfPages(pdf_path) as pdf:
        salvar_pagina_texto(
            pdf,
            titulo=f'Variante {nome} ({perfil["nome"]}) — Etapa 1: Exploração',
            subtitulo=f'Categoria_Id = {categoria}  |  Cachê: {perfil["faixa"]}  |  {dataset_path.name}',
            linhas=[
                f'Perfil: {perfil["nome"]}',
                f'Faixa de cachê (Valor_dia): {perfil["faixa"]}',
                '',
                f'Registros: {df.shape[0]:,}',
                f'Colunas: {df.shape[1]}',
                f'Período (Ano): {df["Ano"].min()} a {df["Ano"].max()}',
                f'Valor_dia médio: R$ {df[TARGET_COL].mean():,.2f}',
                f'Valor_dia mediano: R$ {df[TARGET_COL].median():,.2f}',
                f'Valor_dia desvio padrão: R$ {df[TARGET_COL].std():,.2f}',
                f'Valor_dia mínimo: R$ {df[TARGET_COL].min():,.2f}',
                f'Valor_dia máximo: R$ {df[TARGET_COL].max():,.2f}',
                f'Fornecedores únicos: {df["Fornecedor_Id"].nunique()}',
                f'Regiões únicas: {df["Regiao_Id"].nunique()}',
                f'Zonas únicas: {df["Zona_Id"].nunique()}',
            ],
        )
        salvar_tabela_pdf(pdf, f'{rotulo} — Estatísticas descritivas', df.describe().T.round(2))

        fig, ax = plt.subplots(figsize=(10, 5))
        sns.histplot(df[TARGET_COL], bins=50, kde=True, ax=ax)
        ax.set_title(f'{rotulo} — Distribuição da variável-alvo Valor_dia')
        ax.set_xlabel('Valor_dia (R$)')
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        # -------------------------------------------------------------------
        # Etapa 2 — Detecção de anomalias (IQR + K-Means)
        # -------------------------------------------------------------------
        mascara_iqr, lim_inf, lim_sup, q1, q3, iqr = detectar_anomalias_iqr(df[TARGET_COL])
        df['anomalia_iqr'] = mascara_iqr

        colunas_anomalia = ['Ano', 'Mes', TARGET_COL]
        scores, limiar_kmeans, _, _ = detectar_anomalias_kmeans(
            df,
            colunas=colunas_anomalia,
            n_clusters=N_CLUSTERS_ANOMALIA,
            percentil=PERCENTIL_ANOMALIA,
            random_state=RANDOM_STATE,
        )
        df['score_anomalia'] = scores
        df['anomalia_kmeans'] = df['score_anomalia'] > limiar_kmeans
        df['anomalia_prevista'] = df['anomalia_iqr'] | df['anomalia_kmeans']

        print(
            f'[{nome}] Anomalias -> IQR: {int(df["anomalia_iqr"].sum())} | '
            f'K-Means: {int(df["anomalia_kmeans"].sum())} | '
            f'Combinadas: {int(df["anomalia_prevista"].sum())} '
            f'({df["anomalia_prevista"].mean():.1%})'
        )

        salvar_pagina_texto(
            pdf,
            titulo=f'Variante {nome} ({perfil["nome"]}) — Etapa 2: Detecção de Anomalias',
            linhas=[
                f'Anomalias IQR: {int(df["anomalia_iqr"].sum())}',
                f'Anomalias K-Means: {int(df["anomalia_kmeans"].sum())}',
                f'Anomalias combinadas: {int(df["anomalia_prevista"].sum())}',
                f'Percentual combinado: {df["anomalia_prevista"].mean():.1%}',
                '',
                'Regra IQR:',
                f'  Limite inferior = {lim_inf:,.2f}',
                f'  Limite superior = {lim_sup:,.2f}',
                '',
                'Regra K-Means:',
                f'  Limiar no percentil {PERCENTIL_ANOMALIA}% = {limiar_kmeans:.4f}',
            ],
        )

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].boxplot(df[TARGET_COL], vert=True)
        axes[0].set_title(f'{rotulo} — Valor_dia COM anomalias')
        axes[0].set_ylabel('Valor_dia (R$)')
        df_sem_iqr = df[~df['anomalia_iqr']]
        axes[1].boxplot(df_sem_iqr[TARGET_COL], vert=True)
        axes[1].set_title(f'{rotulo} — Valor_dia SEM anomalias IQR')
        axes[1].set_ylabel('Valor_dia (R$)')
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        # -------------------------------------------------------------------
        # Etapa 3 — Tratamento
        # -------------------------------------------------------------------
        n_antes = len(df)
        df_tratado = df[~df['anomalia_prevista']].copy()
        n_depois = len(df_tratado)

        df_tratado, _ = tratar_ausentes(df_tratado)

        for coluna in ['Ano', 'Mes', TARGET_COL]:
            df_tratado[coluna] = pd.to_numeric(df_tratado[coluna], errors='coerce')
        for coluna in ['Fornecedor_Id', 'Regiao_Id', 'Zona_Id']:
            df_tratado[coluna] = pd.to_numeric(df_tratado[coluna], errors='coerce').astype('Int64')

        colunas_aux = ['anomalia_iqr', 'anomalia_kmeans', 'score_anomalia', 'anomalia_prevista']
        df_tratado = df_tratado.drop(columns=colunas_aux)

        dataset_tratado_path = DATASETS_DIR / f'Dataset_Secretaria_Cultura_SP_var{nome}_tratado.csv'
        df_tratado.to_csv(dataset_tratado_path, sep=';', decimal=',', index=False)
        print(
            f'[{nome}] Tratamento -> {n_antes:,} registros | removidos: {n_antes - n_depois:,} | '
            f'finais: {n_depois:,} | salvo: {dataset_tratado_path.name}'
        )

        salvar_pagina_texto(
            pdf,
            titulo=f'Variante {nome} ({perfil["nome"]}) — Etapa 3: Tratamento do Dataset',
            linhas=[
                f'Registros originais: {n_antes:,}',
                f'Anomalias removidas: {n_antes - n_depois:,}',
                f'Registros finais: {n_depois:,}',
                f'Redução: {(n_antes - n_depois) / n_antes:.1%}',
                '',
                f'Valor_dia médio após tratamento: R$ {df_tratado[TARGET_COL].mean():,.2f}',
                f'Valor_dia mediano após tratamento: R$ {df_tratado[TARGET_COL].median():,.2f}',
                '',
                f'Arquivo limpo: {dataset_tratado_path.name}',
            ],
        )

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].hist(df[TARGET_COL], bins=50, color='tomato', alpha=0.7)
        axes[0].set_title(f'{rotulo} — Valor_dia ANTES do tratamento')
        axes[0].set_xlabel('Valor_dia (R$)')
        axes[1].hist(df_tratado[TARGET_COL], bins=50, color='steelblue', alpha=0.7)
        axes[1].set_title(f'{rotulo} — Valor_dia DEPOIS do tratamento')
        axes[1].set_xlabel('Valor_dia (R$)')
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        # -------------------------------------------------------------------
        # Etapa 4 — Preparação
        # -------------------------------------------------------------------
        X_treino, X_teste, y_treino, y_teste = montar_dataset_modelagem(
            df_tratado,
            target_col=TARGET_COL,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
        )

        scaler = StandardScaler()
        X_treino_scaled = scaler.fit_transform(X_treino)
        X_teste_scaled = scaler.transform(X_teste)
        print(f'[{nome}] Preparação -> treino: {X_treino.shape} | teste: {X_teste.shape}')

        resumo_features = pd.DataFrame({
            'informacao': [
                'Variante', 'Perfil', 'Faixa de cachê', 'Categoria_Id',
                'Registros de treino', 'Registros de teste',
                'Total de features', 'Target encoding', 'One-hot encoding', 'Padronização',
            ],
            'valor': [
                nome, perfil['nome'], perfil['faixa'], categoria,
                len(X_treino), len(X_teste),
                X_treino.shape[1], 'Fornecedor_Id', 'Regiao_Id, Zona_Id',
                'StandardScaler (fit no treino)',
            ],
        })
        salvar_tabela_pdf(pdf, f'{rotulo} — Resumo da preparação', resumo_features)

        # -------------------------------------------------------------------
        # Etapa 5 — Treino
        # -------------------------------------------------------------------
        modelo = LinearRegression()
        modelo.fit(X_treino_scaled, y_treino)

        coeficientes = pd.DataFrame({
            'feature': X_treino.columns,
            'coeficiente': modelo.coef_,
        }).assign(abs_coef=lambda d: d['coeficiente'].abs()).sort_values('abs_coef', ascending=False)
        top15 = coeficientes.head(15)

        salvar_pagina_texto(
            pdf,
            titulo=f'Variante {nome} ({perfil["nome"]}) — Etapa 5: Treinamento',
            linhas=[
                f'Intercepto: {modelo.intercept_:,.2f}',
                f'Número de features: {X_treino.shape[1]}',
                f'Registros de treino: {len(X_treino):,}',
                '',
                'Modelo: sklearn.linear_model.LinearRegression',
                'Padronização: StandardScaler',
            ],
        )

        fig, ax = plt.subplots(figsize=(10, 6))
        cores = ['tomato' if c < 0 else 'steelblue' for c in top15['coeficiente']]
        ax.barh(top15['feature'].astype(str), top15['coeficiente'], color=cores)
        ax.set_title(f'{rotulo} — Top 15 coeficientes da Regressão Linear')
        ax.set_xlabel('Valor do coeficiente')
        ax.invert_yaxis()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        # -------------------------------------------------------------------
        # Etapa 6 — Avaliação
        # -------------------------------------------------------------------
        df_metricas, y_pred_treino, y_pred_teste = avaliar_regressao(
            modelo, X_treino_scaled, X_teste_scaled, y_treino, y_teste,
        )
        r2_treino = float(df_metricas.loc[0, 'R2'])
        r2_teste = float(df_metricas.loc[1, 'R2'])
        diff_r2 = r2_treino - r2_teste

        print(f'[{nome}] Avaliação -> R² treino: {r2_treino:.4f} | R² teste: {r2_teste:.4f}')

        salvar_tabela_pdf(pdf, f'{rotulo} — Métricas: Treino vs. Teste', df_metricas.round(4))

        fig, ax = plt.subplots(figsize=(8, 8))
        ax.scatter(y_teste, y_pred_teste, alpha=0.4, edgecolors='k', linewidth=0.3)
        limite = max(y_teste.max(), y_pred_teste.max())
        ax.plot([0, limite], [0, limite], 'r--', label='Predição perfeita')
        ax.set_xlabel('Valor Real (R$)')
        ax.set_ylabel('Valor Previsto (R$)')
        ax.set_title(f'{rotulo} — Teste: Real vs. Previsto')
        ax.legend()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        residuos_teste = y_teste - y_pred_teste
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].scatter(y_pred_teste, residuos_teste, alpha=0.4)
        axes[0].axhline(0, color='red', linestyle='--')
        axes[0].set_xlabel('Valor Previsto (R$)')
        axes[0].set_ylabel('Resíduo (R$)')
        axes[0].set_title(f'{rotulo} — Resíduos vs. Previsto (Teste)')
        axes[1].hist(residuos_teste, bins=40, color='steelblue', edgecolor='white')
        axes[1].set_xlabel('Resíduo (R$)')
        axes[1].set_title(f'{rotulo} — Distribuição dos resíduos (Teste)')
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

        salvar_pagina_texto(
            pdf,
            titulo=f'Variante {nome} ({perfil["nome"]}) — Diagnóstico Final',
            subtitulo=f'Cachê: {perfil["faixa"]}',
            linhas=[
                f'R² Treino: {r2_treino:.4f}',
                f'R² Teste:  {r2_teste:.4f}',
                f'MAE Teste: R$ {df_metricas.loc[1, "MAE"]:,.2f}',
                f'RMSE Teste: R$ {df_metricas.loc[1, "RMSE"]:,.2f}',
                f'Diferença R² (treino - teste): {diff_r2:.4f}',
            ],
        )

    print(f'[{nome}] PDF gerado: {pdf_path.name}')

    return {
        'Variante': nome,
        'Perfil': perfil['nome'],
        'Faixa_cache': perfil['faixa'],
        'Categoria_Id': categoria,
        'Registros_orig': n_antes,
        'Registros_final': n_depois,
        'Features': int(X_treino.shape[1]),
        'Valor_medio': float(df_tratado[TARGET_COL].mean()),
        'R2_treino': r2_treino,
        'R2_teste': r2_teste,
        'MAE_teste': float(df_metricas.loc[1, 'MAE']),
        'RMSE_teste': float(df_metricas.loc[1, 'RMSE']),
        'Diff_R2': diff_r2,
        '_pdf': pdf_path,
    }


print('Função de pipeline pronta.')

Função de pipeline pronta.


## 4. Execução das três variantes (G1, G2, G3)

Agora rodamos o mesmo pipeline para cada variante. O resultado de cada execução é guardado
em `resultados`, que vira a base da comparação final.

In [4]:
resultados = []
for nome, caminho in VARIANTES.items():
    resultados.append(executar_pipeline_completo(nome, caminho))

df_comparativo = pd.DataFrame(
    [{chave: valor for chave, valor in r.items() if chave != '_pdf'} for r in resultados]
)

print('\n=== Resumo comparativo (treino/teste por variante) ===')
display(df_comparativo.round(4))


========================= Variante G1 =========================
[G1] Pequenos Artistas (R$ 0,00 a R$ 9.999,99) | Registros: 4,152 | Categoria_Id: 1


[G1] Anomalias -> IQR: 0 | K-Means: 205 | Combinadas: 205 (4.9%)
[G1] Tratamento -> 4,152 registros | removidos: 205 | finais: 3,947 | salvo: Dataset_Secretaria_Cultura_SP_varG1_tratado.csv


[G1] Preparação -> treino: (3354, 41) | teste: (593, 41)


[G1] Avaliação -> R² treino: 0.4471 | R² teste: 0.2615


[G1] PDF gerado: G1_pipeline.pdf

========================= Variante G2 =========================
[G2] Artistas Médios (R$ 10.000,00 a R$ 49.999,99) | Registros: 1,593 | Categoria_Id: 2


...

[G3] Anomalias -> IQR: 90 | K-Means: 64 | Combinadas: 114 (9.0%)
[G3] Tratamento -> 1,267 registros | removidos: 114 | finais: 1,153 | salvo: Dataset_Secretaria_Cultura_SP_varG3_tratado.csv


[G3] Preparação -> treino: (980, 40) | teste: (173, 40)


[G3] Avaliação -> R² treino: 0.4705 | R² teste: -0.0737


[G3] PDF gerado: G3_pipeline.pdf

=== Resumo comparativo (treino/teste por variante) ===


,Variante,Perfil,Faixa_cache,Categoria_Id,Registros_orig,Registros_final,Features,Valor_medio,R2_treino,R2_teste,MAE_teste,RMSE_teste,Diff_R2
0,G1,Pequenos Artistas,"R$ 0,00 a R$ 9.999,99",1,4152,3947,41,5092.4725,0.4471,0.2615,1490.2473,1860.0338,0.1856
1,G2,Artistas Médios,"R$ 10.000,00 a R$ 49.999,99",2,1593,1531,40,24565.1610,0.4857,0.0973,7528.7374,9384.9041,0.3884
2,G3,Super Artistas,"acima de R$ 50.000,00",3,1267,1153,40,105069.6803,0.4705,-0.0737,47612.1986,59397.6489,0.5442


## 5. Comparação dos resultados

Aqui comparamos as três variantes lado a lado:

- **R² (treino vs. teste)** — capacidade explicativa e sinal de overfitting;
- **MAE no teste** — erro médio absoluto, em reais;
- **RMSE no teste** — erro que penaliza mais os grandes desvios.

A melhor variante para a Regressão Linear é a que combina **R² de teste mais alto** com
**MAE/RMSE mais baixos** e pequena diferença entre treino e teste.

In [5]:
# ---------------------------------------------------------------------------
# Identifica a melhor variante por R² de teste
# ---------------------------------------------------------------------------
melhor = df_comparativo.loc[df_comparativo['R2_teste'].idxmax()]
print(f'Melhor R² de teste: {melhor["Variante"]} - {melhor["Perfil"]} (R² = {melhor["R2_teste"]:.4f})')
print(f'Menor MAE de teste: {df_comparativo.loc[df_comparativo["MAE_teste"].idxmin(), "Variante"]}')
print(f'Menor RMSE de teste: {df_comparativo.loc[df_comparativo["RMSE_teste"].idxmin(), "Variante"]}')

# ---------------------------------------------------------------------------
# PDF comparativo
# ---------------------------------------------------------------------------
pdf_comparativo = RELATORIOS_DIR / '00_comparativo.pdf'
nomes = df_comparativo['Variante'].tolist()
# Rótulos com perfil para deixar os gráficos mais palpáveis
rotulos = [f'{r["Variante"]}\n{r["Perfil"]}' for r in resultados]
x = np.arange(len(nomes))

with PdfPages(pdf_comparativo) as pdf:
    # Página de legenda das categorias
    salvar_pagina_texto(
        pdf,
        titulo='Legenda das categorias de artista',
        subtitulo='Cada variante corresponde a uma faixa de cachê (Valor_dia)',
        linhas=[
            'G1 - Pequenos Artistas : R$ 0,00 a R$ 9.999,99',
            'G2 - Artistas Médios   : R$ 10.000,00 a R$ 49.999,99',
            'G3 - Super Artistas    : acima de R$ 50.000,00',
        ],
    )

    salvar_tabela_pdf(
        pdf,
        'Comparativo das variantes G1, G2 e G3',
        df_comparativo.drop(columns=['Faixa_cache']).round(4),
    )

    # R² treino vs. teste
    fig, ax = plt.subplots(figsize=(10, 6))
    largura = 0.35
    ax.bar(x - largura / 2, df_comparativo['R2_treino'], largura, label='R² Treino', color='steelblue')
    ax.bar(x + largura / 2, df_comparativo['R2_teste'], largura, label='R² Teste', color='darkorange')
    ax.set_xticks(x)
    ax.set_xticklabels(rotulos)
    ax.set_ylabel('R²')
    ax.set_title('R² por categoria de artista (treino vs. teste)')
    ax.legend()
    for i in x:
        ax.text(i - largura / 2, df_comparativo['R2_treino'].iloc[i], f'{df_comparativo["R2_treino"].iloc[i]:.2f}', ha='center', va='bottom', fontsize=8)
        ax.text(i + largura / 2, df_comparativo['R2_teste'].iloc[i], f'{df_comparativo["R2_teste"].iloc[i]:.2f}', ha='center', va='bottom', fontsize=8)
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    # MAE e RMSE no teste
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].bar(rotulos, df_comparativo['MAE_teste'], color='darkorange')
    axes[0].set_title('MAE no teste (R$)')
    axes[0].set_ylabel('R$')
    for i in x:
        axes[0].text(i, df_comparativo['MAE_teste'].iloc[i], f'{df_comparativo["MAE_teste"].iloc[i]:,.0f}', ha='center', va='bottom', fontsize=8)
    axes[1].bar(rotulos, df_comparativo['RMSE_teste'], color='tomato')
    axes[1].set_title('RMSE no teste (R$)')
    axes[1].set_ylabel('R$')
    for i in x:
        axes[1].text(i, df_comparativo['RMSE_teste'].iloc[i], f'{df_comparativo["RMSE_teste"].iloc[i]:,.0f}', ha='center', va='bottom', fontsize=8)
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    salvar_pagina_texto(
        pdf,
        titulo='Conclusão — Comparação por categoria de artista',
        linhas=[
            f'Melhor R² de teste: {melhor["Variante"]} - {melhor["Perfil"]} ({melhor["R2_teste"]:.4f})',
            '',
            'Leitura:',
            '- R² de teste mais alto => modelo explica melhor o Valor_dia.',
            '- MAE/RMSE menores => erros em reais menores.',
            '- Diferença treino-teste alta => sinal de overfitting.',
            '',
        ] + [
            f'{r["Variante"]} ({r["Perfil"]}): R2_teste={r["R2_teste"]:.3f} | '
            f'MAE={r["MAE_teste"]:,.0f} | RMSE={r["RMSE_teste"]:,.0f}'
            for r in resultados
        ],
    )

print(f'PDF gerado: {pdf_comparativo.name}')

Melhor R² de teste: G1 - Pequenos Artistas (R² = 0.2615)
Menor MAE de teste: G1
Menor RMSE de teste: G1


PDF gerado: 00_comparativo.pdf


## 6. Consolidação dos relatórios em PDF

Por fim, juntamos o comparativo e o PDF de cada variante em um único arquivo `00_consolidado.pdf`.

In [6]:
pdfs_finais = [pdf_comparativo] + [r['_pdf'] for r in resultados]

pdf_consolidado = RELATORIOS_DIR / '00_consolidado.pdf'
consolidar_pdfs(pdfs_finais, pdf_consolidado)

print('\n=== Pipeline concluído com sucesso ===')
print('\nPDFs gerados:')
for pdf in pdfs_finais + [pdf_consolidado]:
    status = 'OK' if pdf.exists() else 'FALTANDO'
    print(f'  [{status}] {pdf.name}')

PDF consolidado salvo em: 00_consolidado.pdf

=== Pipeline concluído com sucesso ===

PDFs gerados:
  [OK] 00_comparativo.pdf
  [OK] G1_pipeline.pdf
  [OK] G2_pipeline.pdf
  [OK] G3_pipeline.pdf
  [OK] 00_consolidado.pdf


## 7. Observações finais

### Legenda das categorias

- **G1 — Pequenos Artistas**: cachê de R$ 0,00 a R$ 9.999,99 (`Categoria_Id == 1`);
- **G2 — Artistas Médios**: cachê de R$ 10.000,00 a R$ 49.999,99 (`Categoria_Id == 2`);
- **G3 — Super Artistas**: cachê acima de R$ 50.000,00 (`Categoria_Id == 3`).

### Notas

- Cada variante (G1, G2, G3) corresponde a uma `Categoria_Id` distinta. Como essa coluna é
  constante dentro de cada arquivo, ela é removida antes da modelagem (junto com `Id`).
- O esquema de features é idêntico nas três variantes (target encoding em `Fornecedor_Id`,
  one-hot em `Regiao_Id` e `Zona_Id`, numéricas `Ano` e `Mes`), o que torna a comparação justa.
- Os datasets tratados de cada variante são salvos como
  `Dataset_Secretaria_Cultura_SP_varG1_tratado.csv`, `...varG2_tratado.csv` e `...varG3_tratado.csv`.
- A tabela e os gráficos de `00_comparativo.pdf` mostram em qual categoria a Regressão Linear
  obtém melhor desempenho (maior R² de teste e menores MAE/RMSE).